# Lesson 8 — Measure how long a dislocation survives before building an executor

Every coherence violation has a lifetime. If the median is shorter than the round trip, it is not an opportunity — it is a data artefact, and the race was lost before it was entered.

**The rule.** `survival S(t) = P(violation still open after t seconds)`

**When it holds.** Once enough episodes have been recorded to estimate a median. Until then the honest answer is that there is no estimate.

**When it fails.** Reading a short half-life as 'be faster'. Against commercial detection in tens of milliseconds over REST polling, the edge has to be in structures nobody scans for, not in speed.

| | |
|---|---|
| Lesson id | `halflife` |
| Pane it appears on | `diffusion` (panes carry more than one lesson) |
| Code it is about | `modules/coherence/episodes.py` |
| Tests that go red if it stops being true | `tests/test_coherence_episodes.py` |
| Pane shipped | yes |

Every cell below runs against the real kernel. Nothing here is a re-implementation:
a number this notebook prints is the number the engine would produce for the same
input. The recorded Kalshi payloads come from `tests/fixtures/coherence/`.

In [ ]:
import json
import sys
from decimal import Decimal
from pathlib import Path

# This notebook lives in notebooks/coherence_lab/ and imports the kernel two
# levels up. Found by walking upward rather than by counting parents, so the
# notebook runs from its own directory or from Part2_Infrastructure.
HERE = Path.cwd().resolve()
ROOT = next((path for path in (HERE, *HERE.parents) if (path / "modules" / "coherence" / "kernel").is_dir()), None)
if ROOT is None:
    raise SystemExit(f"no coherence kernel above {HERE}: open this notebook from inside Part2_Infrastructure")
sys.path.insert(0, str(ROOT))

FIXTURES = ROOT / "tests" / "fixtures" / "coherence"


def fixture(name: str) -> dict:
    """One recorded Kalshi response, envelope and all, exactly as it was sent.

    These are captures, not mocks. Where a number below looks odd it is because
    the exchange quoted it, and `tools/capture_kalshi_fixtures.py` re-records
    them.
    """
    return json.loads((FIXTURES / f"{name}.json").read_text(encoding="utf-8"))


print(f"kernel root       {ROOT}")
print(f"recorded fixtures {FIXTURES.is_dir()}")

## 1. A poll tape, and the episodes it opens and closes

In [ ]:
from modules.coherence.episodes import (
    MIN_EPISODES_FOR_HALF_LIFE,
    POLLS_TO_CLOSE,
    EpisodeTracker,
    survival,
    verdict_for,
)

POLL_INTERVAL_NS = 1_000_000_000  # a one-second poll, which is what a REST token budget buys


def replay(runs, interval_ns=POLL_INTERVAL_NS):
    """Replay a poll tape. Each entry is how many consecutive polls saw the violation.

    Polls, not seconds: a lifetime is only ever observed in multiples of the
    cadence that observed it, and writing the tape in polls keeps that visible.
    """
    tracker = EpisodeTracker()
    stamp = 0
    for index, polls in enumerate(runs):
        for _ in range(polls):
            tracker.observe(
                f"E-{index}", "KXDEMO", f"E-{index}", 0, stamp, True,
                family="additive", ci=Decimal("0.0300"), net_edge=Decimal("0.4000"),
            )
            stamp += interval_ns
        for _ in range(POLLS_TO_CLOSE):
            tracker.observe(f"E-{index}", "KXDEMO", f"E-{index}", 0, stamp, False)
            stamp += interval_ns
    return tracker


FAST = [1, 1, 2, 1, 3, 1, 2, 1, 1, 2, 1, 4]
fast = replay(FAST)
print(f"  {len(fast.closed)} episodes closed, {len(fast.open_episodes)} still open")
for episode in fast.closed[:4]:
    print(f"    {episode.component_id} lasted {episode.lifetime_s}s, peak ci {episode.peak_ci}")
print()
print(f"  Two coherent polls close an episode, not one (POLLS_TO_CLOSE = {POLLS_TO_CLOSE}).")
print("  A single poll can miss a violation because one leg's book was momentarily")
print("  unreadable, and closing on that would cut long episodes into strings of short")
print("  ones — biasing the median DOWN, which makes the exchange look faster than it is.")

## 2. Below the sample floor, the median is withheld

In [ ]:
thin = survival(replay([2, 3, 5]).closed)
print(f"  episodes {thin.episodes}")
print(f"  median   {thin.median_s!r}")
print(f"  reason   {thin.reason}")
print()
print(f"  The curve is still drawn — {len(thin.points)} real points — and only the summary")
print(f"  statistic a reader would quote is withheld until there are {MIN_EPISODES_FOR_HALF_LIFE} of them.")
print("  Open episodes are excluded entirely rather than counted at their current age: an")
print("  episode still running is a lower bound on a lifetime, not a measurement of one.")

## 3. The survival curve

In [ ]:
curve = survival(fast.closed)
print(f"  {curve.episodes} closed episodes, median {curve.median_s}s")
print()
print("  t (s)     surviving")
for seconds, surviving in curve.points:
    bar = "#" * int(surviving * 40)
    print(f"  {seconds:>7}   {surviving}  {bar}")

## 4. The verdict against a round trip

In [ ]:
for round_trip in (Decimal("0.35"), Decimal("2"), Decimal("60")):
    print(f"  round trip {round_trip}s:")
    print(f"    {verdict_for(curve, round_trip)}")
print()
slow = survival(replay([90, 120, 140, 200, 240, 260, 300, 420, 600]).closed)
print(f"  a slower series, median {slow.median_s}s:")
print(f"    {verdict_for(slow, Decimal('2'))}")
print()
quick = survival(replay([1, 2, 1, 3, 1, 1, 2, 1, 4, 1], interval_ns=50_000_000).closed)
print(f"  the same shape of tape polled every 50ms, median {quick.median_s}s:")
print(f"    {verdict_for(quick, Decimal('0.35'))}")
print()
print("  That last line is also the censoring, stated out loud: a REST-polled tape cannot")
print("  record a dislocation shorter than its own cadence. The one-second tape above has")
print("  no episode under two seconds because it could not have had one, and a median read")
print("  off it is a statement about the poller as much as about the exchange.")
print()
print("  A short half-life is not an instruction to be faster. Against commercial")
print("  detection in tens of milliseconds over REST polling, the race was lost before it")
print("  was entered, and the edge has to be in structures nobody scans for.")